In [ ]:
%pip install lxml bs4 langchain langchain-openai faiss-cpu networkx matplotlib
dbutils.library.restartPython()

In [ ]:
import re
import networkx as nx
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from pyspark.sql import Row

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.schema import Document

In [ ]:
xml_paths = [
    "/Volumes/...3829c0e9.../WebHome.xml",
    "/Volumes/...76f049ea.../WebHome.xml",
    "/Volumes/...07730902.../WebHome.xml",
    "/Volumes/...f2280f6d.../WebHome.xml",
    "/Volumes/...96bf76a9.../WebHome.xml"
]

rows = []

for path in xml_paths:
    with open(path, "rb") as f:
        soup = BeautifulSoup(f.read(), "xml")

    content = soup.find("content").text if soup.find("content") else ""

    rows.append({
        "content": content,
        "source": path
    })

print("Loaded docs:", len(rows))

In [ ]:
def clean_text(text):
    text = re.sub(r"<!\[CDATA\[|\]\]>", "", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

for r in rows:
    r["content"] = clean_text(r["content"])

In [ ]:
G = nx.DiGraph()

# 🔹 Step 1: Map doc_name → source
docname_to_source = {}

for r in rows:
    source = r["source"]
    doc_name = source.split("/")[-2]   # folder name
    docname_to_source[doc_name] = source

# 🔹 Step 2: Add nodes
for r in rows:
    G.add_node(r["source"], content=r["content"])

# 🔹 Step 3: Extract links from content
for r in rows:
    src = r["source"]
    content = r["content"]

    # 🔥 Extract links like: doc:Temp_Import.xxx
    links = re.findall(r"doc:([A-Za-z0-9._-]+)", content)

    for link in links:
        for key in docname_to_source:
            if link in key:
                target = docname_to_source[key]

                if src != target:
                    G.add_edge(src, target, relation="links_to")

print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))
print("Edges List:", list(G.edges()))

In [ ]:
pagerank = nx.pagerank(G)

for node in G.nodes():
    G.nodes[node]["pagerank"] = pagerank.get(node, 0)

print("PageRank done")

In [ ]:
embedding_model = OpenAIEmbeddings()

documents = []

for r in rows:
    documents.append(
        Document(
            page_content=r["content"],
            metadata={"source": r["source"]}
        )
    )

vector_store = FAISS.from_documents(documents, embedding_model)

print("Vector store ready")

In [ ]:
def graph_retriever(graph, vector_store, query, max_hops=2, top_k=3):

    docs = vector_store.similarity_search(query, k=top_k)

    start_nodes = [d.metadata["source"] for d in docs]
    print("Start Nodes:", start_nodes)

    visited = set()
    scores = {}
    path = []

    queue = [(node, 0) for node in start_nodes]

    for node in start_nodes:
        scores[node] = graph.nodes[node]["pagerank"]

    while queue:
        current, hop = queue.pop(0)

        if hop >= max_hops or current in visited:
            continue

        visited.add(current)
        path.append(current)

        for neighbor in graph.neighbors(current):

            pr = graph.nodes[neighbor]["pagerank"]
            score = pr / (hop + 1)

            if neighbor not in scores or score > scores[neighbor]:
                scores[neighbor] = score

            queue.append((neighbor, hop + 1))

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_nodes = [n for n, _ in ranked[:top_k]]

    print("Top Nodes:", top_nodes)

    return top_nodes, path

In [ ]:
def visualize_graph(graph, path=None):

    plt.figure(figsize=(10, 7))
    pos = nx.circular_layout(graph)

    labels = {n: n.split("/")[-2] for n in graph.nodes()}

    nx.draw(graph, pos,
            labels=labels,
            node_color="lightblue",
            node_size=2000)

    nx.draw_networkx_edges(
        graph, pos,
        edge_color="gray",
        arrows=True,
        connectionstyle='arc3,rad=0.2'
    )

    if path and len(path) > 1:
        edges = list(zip(path, path[1:]))

        nx.draw_networkx_edges(
            graph, pos,
            edgelist=edges,
            edge_color="red",
            width=3
        )

    plt.title("GraphRAG RDF Graph")
    plt.show()

In [ ]:
class GraphRAG:

    def __init__(self, graph, vector_store, llm):
        self.graph = graph
        self.vector_store = vector_store
        self.llm = llm

    def answer(self, query):

        nodes, path = graph_retriever(self.graph, self.vector_store, query)

        visualize_graph(self.graph, path)

        context = ""
        for n in nodes:
            context += "\n" + self.graph.nodes[n]["content"]

        prompt = f"""
        Answer using context:

        {context}

        Question: {query}
        """

        return self.llm.invoke(prompt)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag = GraphRAG(G, vector_store, llm)

response = rag.answer("What is connection between RRDS and windmill event?")

print(response)

In [ ]:
for r in rows:
    print("SOURCE:", r["source"])
    print("LINKS:", re.findall(r"doc:(.*?)\]", r["content"])[:3])